# Phase 8: TOPSIS Weight Sensitivity Analysis

The Phase 7 ranking depends on the weights we chose, and different organisations care about different things. A trading firm might only care about accuracy. A provider with a carbon target might care mostly about emissions. Here we re-run TOPSIS with four quite different weightings and check **whether the recommendation changes**. If the same model comes first every time, the recommendation is robust. If it changes, we can say exactly which priorities favour which model.

| Scenario | MAE | RMSE | MAPE | R² | Train time | CO₂ | Who it represents |
|---|---|---|---|---|---|---|---|
| Primary (Phase 7) | 0.15 | 0.15 | 0.15 | 0.20 | 0.15 | 0.20 | our balanced-but-accuracy-led choice |
| Balanced | 1/6 | 1/6 | 1/6 | 1/6 | 1/6 | 1/6 | no opinion: every criterion equal |
| Accuracy-Priority | 0.20 | 0.20 | 0.15 | 0.25 | 0.10 | 0.10 | prediction quality above all (accuracy = 0.80) |
| Energy-Priority | 0.10 | 0.10 | 0.10 | 0.10 | 0.15 | 0.45 | strict carbon budget (CO₂ alone = 0.45) |
| Speed-Priority | 0.10 | 0.10 | 0.10 | 0.10 | 0.45 | 0.15 | frequent retraining / fast turnaround (time = 0.45) |

The weights are defined in `scripts/config.py` (`TOPSIS_SCENARIOS`). Each scenario's weights add up to 1.

As an **extra check** we also sweep the weights continuously: the total weight on cost (time + CO₂) goes from 0% to 100% in 5% steps, with the rest spread over the accuracy criteria in the primary proportions. That shows the exact point at which the winner changes, rather than only looking at four fixed scenarios.

We pay particular attention to the **LSTM's rank**, because whether deep learning is worth its cost is a central question of the paper.

**Run time:** a few seconds. It's only arithmetic on the Phase 7 decision matrix.

## 0. Setup

In [1]:
# Mount Google Drive (Colab) or use the local Drive folder (PC), then load our shared code.
import os, sys, importlib

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Green_ML_Project'
except ImportError:
    PROJECT_ROOT = r'H:\My Drive\Green_ML_Project'   # running on the PC instead of Colab

os.environ['GREEN_ML_ROOT'] = PROJECT_ROOT
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'scripts'))

import config as C
import utils as U
importlib.reload(C); importlib.reload(U)   # picks up any edits to the scripts without restarting

print('Project folder:', C.PROJECT_ROOT)
print('Folder found:  ', os.path.isdir(C.PROJECT_ROOT))

Mounted at /content/drive
Project folder: /content/drive/MyDrive/Green_ML_Project
Folder found:   True


In [2]:
import numpy as np
import pandas as pd
import model_definitions as M
importlib.reload(M)

pd.set_option('display.width', 180)
log = U.get_logger('phase08_sensitivity_analysis')

decision = pd.read_csv(C.TABLES_DIR / 'phase7_decision_matrix.csv', index_col='Model')
benefit = C.TOPSIS_CRITERIA
scenarios = {'Primary': C.TOPSIS_PRIMARY_WEIGHTS, **C.TOPSIS_SCENARIOS}
for s, w in scenarios.items():
    assert abs(sum(w.values()) - 1) < 1e-6, s
    log.info(f'{s:<18}: ' + ', '.join(f'{k}={v:.3f}' for k, v in w.items()))

RUN STARTED 2026-10-04 07:28:30 | phase08_sensitivity_analysis
python 3.13.15 | pandas 2.2.3 | numpy 2.1.3 | machine: Linux-6.6.122+-x86_64-with-glibc2.39
project root: /content/drive/MyDrive/Green_ML_Project
Primary           : MAE=0.150, RMSE=0.150, MAPE=0.150, R2=0.200, train_time_sec=0.150, co2_emissions_kg=0.200
Balanced          : MAE=0.167, RMSE=0.167, MAPE=0.167, R2=0.167, train_time_sec=0.167, co2_emissions_kg=0.167
Accuracy-Priority : MAE=0.200, RMSE=0.200, MAPE=0.150, R2=0.250, train_time_sec=0.100, co2_emissions_kg=0.100
Energy-Priority   : MAE=0.100, RMSE=0.100, MAPE=0.100, R2=0.100, train_time_sec=0.150, co2_emissions_kg=0.450
Speed-Priority    : MAE=0.100, RMSE=0.100, MAPE=0.100, R2=0.100, train_time_sec=0.450, co2_emissions_kg=0.150


## 1. TOPSIS under each scenario

In [3]:
closeness, ranks = {}, {}
for s, w in scenarios.items():
    r = U.topsis(decision, w, benefit)
    closeness[s], ranks[s] = r['closeness'], r['rank']
closeness = pd.DataFrame(closeness).loc[decision.index]
ranks = pd.DataFrame(ranks).loc[decision.index]

U.log_df(log, closeness.round(4), 'CLOSENESS COEFFICIENT PER SCENARIO (higher is better)')
U.log_df(log, ranks, 'RANK PER SCENARIO (1 = recommended)')

table = pd.DataFrame({'Model': decision.index})
for s in scenarios:
    table[f'{s} C'] = closeness[s].values
    table[f'{s} rank'] = ranks[s].values
table.to_csv(C.TABLES_DIR / 'phase8_sensitivity_raw.csv', index=False)   # exact numbers (Phase 10 reads this)

compact = pd.DataFrame({'Model': decision.index,
                        **{s: [f'{c:.3f} (#{k})' for c, k in zip(closeness[s], ranks[s])] for s in scenarios}})
U.save_table(compact, 'phase8_sensitivity', 'TOPSIS closeness (rank) under each weighting scenario')
compact

--- CLOSENESS COEFFICIENT PER SCENARIO (higher is better) ---
                   Primary  Balanced  Accuracy-Priority  Energy-Priority  Speed-Priority
Model                                                                                   
Linear Regression   0.8688    0.8499             0.7631           0.9492          0.9489
Decision Tree       0.7620    0.7308             0.6234           0.9012          0.9004
Random Forest       0.2201    0.2496             0.3548           0.0901          0.0905
XGBoost             0.9683    0.9657             0.9564           0.9756          0.9706
LightGBM            0.9652    0.9654             0.9686           0.9647          0.9617
LSTM                0.2286    0.2382             0.2755           0.2013          0.1915
--- RANK PER SCENARIO (1 = recommended) ---
                   Primary  Balanced  Accuracy-Priority  Energy-Priority  Speed-Priority
Model                                                                                   
Line

,Model,Primary,Balanced,Accuracy-Priority,Energy-Priority,Speed-Priority
0,Linear Regression,0.869 (#3),0.850 (#3),0.763 (#3),0.949 (#3),0.949 (#3)
1,Decision Tree,0.762 (#4),0.731 (#4),0.623 (#4),0.901 (#4),0.900 (#4)
2,Random Forest,0.220 (#6),0.250 (#5),0.355 (#5),0.090 (#6),0.091 (#6)
3,XGBoost,0.968 (#1),0.966 (#1),0.956 (#2),0.976 (#1),0.971 (#1)
4,LightGBM,0.965 (#2),0.965 (#2),0.969 (#1),0.965 (#2),0.962 (#2)
5,LSTM,0.229 (#5),0.238 (#6),0.276 (#6),0.201 (#5),0.191 (#5)


## 2. Does the recommendation change?

In [4]:
winners = {s: ranks[s].idxmin() for s in scenarios}
for s, m in winners.items():
    log.info(f'{s:<18}: recommended = {m}')
same = len(set(winners.values())) == 1
log.info(f'Same model recommended in every scenario: {same}')

rank_range = ranks.max(axis=1) - ranks.min(axis=1)
for m in decision.index:
    log.info(f'{m:<17}: rank {ranks.loc[m].min()}..{ranks.loc[m].max()} across scenarios '
             f'(moves {rank_range[m]} places) | ranks: {ranks.loc[m].astype(int).to_dict()}')

lstm = ranks.loc['LSTM']
log.info(f'LSTM ranks: {lstm.astype(int).to_dict()} -> best position {lstm.min()} of {len(decision)}')

Primary           : recommended = XGBoost
Balanced          : recommended = XGBoost
Accuracy-Priority : recommended = LightGBM
Energy-Priority   : recommended = XGBoost
Speed-Priority    : recommended = XGBoost
Same model recommended in every scenario: False
Linear Regression: rank 3..3 across scenarios (moves 0 places) | ranks: {'Primary': 3, 'Balanced': 3, 'Accuracy-Priority': 3, 'Energy-Priority': 3, 'Speed-Priority': 3}
Decision Tree    : rank 4..4 across scenarios (moves 0 places) | ranks: {'Primary': 4, 'Balanced': 4, 'Accuracy-Priority': 4, 'Energy-Priority': 4, 'Speed-Priority': 4}
Random Forest    : rank 5..6 across scenarios (moves 1 places) | ranks: {'Primary': 6, 'Balanced': 5, 'Accuracy-Priority': 5, 'Energy-Priority': 6, 'Speed-Priority': 6}
XGBoost          : rank 1..2 across scenarios (moves 1 places) | ranks: {'Primary': 1, 'Balanced': 1, 'Accuracy-Priority': 2, 'Energy-Priority': 1, 'Speed-Priority': 1}
LightGBM         : rank 1..2 across scenarios (moves 1 places) | 

## 3. Extra check: sweep from "accuracy only" to "cost only"

The x-axis is the share of the total weight given to cost (training time + CO₂, split 15:20 as in the primary weights). The rest goes to accuracy, split 15:15:15:20 as in the primary weights.
- At 0% the ranking is decided by accuracy alone.
- At 100% it's decided by cost alone.
- The primary weights sit at 35%.

The chart for this goes in Phase 10. Here we just find where the winner changes.

In [5]:
acc_split = {'MAE': 0.15, 'RMSE': 0.15, 'MAPE': 0.15, 'R2': 0.20}
cost_split = {'train_time_sec': 0.15, 'co2_emissions_kg': 0.20}
acc_tot, cost_tot = sum(acc_split.values()), sum(cost_split.values())

sweep = []
for share in np.round(np.arange(0, 1.0001, 0.05), 2):
    w = {**{k: (1 - share) * v / acc_tot for k, v in acc_split.items()},
         **{k: share * v / cost_tot for k, v in cost_split.items()}}
    # an all-zero group would leave TOPSIS with zero-weight columns, which is fine mathematically
    r = U.topsis(decision, w, benefit)
    sweep.append({'cost_weight_share': share, **{f'C_{m}': r['closeness'][m] for m in decision.index},
                  'winner': r['closeness'].idxmax()})
sweep = pd.DataFrame(sweep)
sweep.to_csv(C.TABLES_DIR / 'phase8_weight_sweep.csv', index=False)

changes = sweep[sweep['winner'] != sweep['winner'].shift()]
for _, row in changes.iterrows():
    log.info(f'From cost share {row.cost_weight_share:.0%} the winner is {row.winner}')
log.info('PHASE 8 COMPLETE')
sweep[['cost_weight_share', 'winner']].T

From cost share 0% the winner is LightGBM
From cost share 35% the winner is XGBoost
From cost share 80% the winner is Linear Regression
PHASE 8 COMPLETE


,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
cost_weight_share,0.0,0.05,0.1,0.15,0.2,0.25,0.3,0.35,0.4,0.45,...,0.55,0.6,0.65,0.7,0.75,0.8,0.85,0.9,0.95,1.0
winner,LightGBM,LightGBM,LightGBM,LightGBM,LightGBM,LightGBM,LightGBM,XGBoost,XGBoost,XGBoost,...,XGBoost,XGBoost,XGBoost,XGBoost,XGBoost,Linear Regression,Linear Regression,Linear Regression,Linear Regression,Linear Regression


## Done: what to check

- **"Same model recommended in every scenario"**: if `True`, the recommendation is robust, which is a strong statement for the paper. If `False`, the paper should say which priorities favour which model. That's still a useful, practical finding.
- **LSTM ranks**: where deep learning lands under each priority.
- **Sweep**: the cost share at which the winner switches. If it's far from our primary 35%, the primary recommendation is safe.

Next: `09_generalization_test.ipynb`.